<h1 style="font-size: 40px; margin-bottom: 0px;">13.1 Differential Expression Analysis</h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 98%;"></hr>

Today, we'll perform differential expression analysis to obtain a <code>.csv</code> file that we can then play around in and explore more using Python. As we're performing the differential expression analysis, we'll break it up into smaller steps to review what Dr. Ingolia taught in lecture to see what's going on under the hood.

<strong>Learning objectives:</strong>

<ul>
    <li>Review differential expression analysis</li>
    <li>Perform hierarchical clustering</li>
    <li>Perform differential expression analysis</li>
    <li>Visualize results as an MA plot</li>
</ul>

<h1 style="font-size: 40px; margin-bottom: 0px;">Load in packages</h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 98%;"></hr>

Like our previous lesson, we'll first load in the packages that we'll need for today's analysis, and then we'll briefly review some of the initial steps of DESeq2 that we did previously to set up for our principal component analysis.

In [ ]:
library(DESeq2)
library(ggplot2)
library(pheatmap)
library(viridis)

<h1 style="font-size: 40px; margin-bottom: 0px;">Prepare your <code>DESeqDataSet</code></h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 98%;"></hr>

Since this is a new notebook, we'll need to bring in our counts matrix and conditions matrix again and use them to create a <code>DESeqDataSet</code> for us to use to do some more clustering. 

Let's bring over our initial code from notebook 12-1 for setting up our <code>DESeqDataSet</code>

<h1 style="font-size: 40px; margin-bottom: 0px;">Perform hierarchical clustering</h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 98%;"></hr>

A way of determining the similarities and dissimilarities of our samples is to perform hierarchical clustering based on our top 500 genes with the highest variance. Recall from Dr. Ingolia's lecture that hierarchical clustering can be thought of as a "bottom-up" approach to clustering where you draw small clusters first, and then begin drawing larger and larger clusters out of those smaller clusters. This then allows you to group samples or conditions based on their gene expression patterns, which are in this case the top 500 most variable genes.

You can see how our data is grouped together in the heatmap, and we can see large blocks or groupings in the heatmap, where we have clusters of genes that are overexpressed in our TAZ KO samples and groups of genes that exhibit reduced expression in our KO samples. 

Something to keep in mind is that we supplied the argument <code>scale="row"</code>. This argument centers the mean of the data for each row around 0 and also scales each row's standard deviation to 1. This allows differences within a row to be more apparent. This is helpful when we want to visualize and compare relative changes between samples without having our visualization dominated by a few highly expressed genes.

<h1 style="font-size: 40px; margin-bottom: 0px;">Differential expression analysis</h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 98%;"></hr>

Here, we'll return to our <code>DESeqDataSet</code> to finish up our differential expression analysis. Recall from Dr. Ingolia's lecture that we've already performed some of the intial steps for differential expression analysis, where we estimated the size factors in order to account for differences in sequencing depth.

<h2>Estimate Dispersions</h2>

The next step is to then estimate the spread of our measurements, otherwise referred to as the dispersion. DESeq2 calculates the estimated dispersion (&#593;) as a function of the mean (&micro;) and variance. In other words, the estimated dispersion is the expected spread of the data for a given mean based on your data. This allows DESeq2 to identify what is likely to be true variation in the data resulting from biological or technical effects by shrinking the dispersion  of each gene towards the calculated estimated value for that specific mean. This can be thought of as modeling what the noise is for our experiment in order to distinguish biological and technical differences in our samples from what are just noise in our measurements.

We'll do this by making use of the <code>estimateDispersions()</code> function, which is part of the DESeq2 package.

Like with our other intermediate calculations, we place the output into our <code>DESeqDataSet</code>.

Then we can visualize the dispersion estimate using the <code>plotDispEsts()</code> function.

In the plot, each dot corresponds to a single gene plotted with its mean on the X-axis and its calculated dispersion on the Y-axis. The red line is the estimated dispersion based on your whole dataset with an estimated dispersion for each expression level. A "good" dispersion plot should have genes following the estimated dispersion with a downward trend that levels out at high expression. The blue indicates the final dispersion after shrinkage, essentially removing what variation may be just due to noise in the measurements. The dots with the genes showing potential biological or technical variation greater than the expected noise marked with a blue outline.

What you would normally expect to see is a fitted line that increases in dispersion as the mean decreases (inversely correlated to mean). This is due to the fact that statistical noise has a greater impact (accounts for more of the variation) when the mean is smaller.

<h2>Hypothesis testing: negative binomial Wald test to determine significance</h2>

Finally, we'll use the Wald test to detect differentially expressed genes and determine if they are significant. DESeq2 models expression based on a negative binomial distribution, and recall from Dr. Ingolia's lecture that the negative binomial distribution can be thought of like a Poisson distribution but able to account for extra variance as a second parameter. 

For hypothesis testing, DESeq2 sets the null hypothesis for each gene as having no difference between sample groups, so no log fold change difference (equal to 0). To test this hypothesis, DESeq2 makes use of the Wald test to compare the sample groups.

To run a Wald test on our samples, we can make use of the <code>nbinomWaldTest()</code> function.

Then we can pull out the results out of our <code>DESeqDataSet</code> using the <code>results()</code> function, and then assigning that to a new variable.

Let's take a look at how our results table looks like:

<h2>Export results of differential expression analysis</h2>

Like with Python, we can export the dataframe containing our results, just with slightly different syntax.

This will output a .csv file containing the results of our differential expression analysis.

<h2>Export rlog transformed counts</h2>

While we're at it, let's also export our rlog transformed counts for use later on.

<h2>Generate an MA Plot</h2>

MA plots visualize the ratio or fold-change (M) on the y-axis plotted against the average or mean normalized counts (A) along the x-axis, and for RNA-seq analyses, these types of plots are commonly used to visualize differential expression levels.

With our differential expression analysis results, we can generate an MA plot from our class dataset. To do this, we'll make use of DESeq2's <code>plotMA()</code> function.

By default, <code>plotMA()</code> highlights genes whose p-adjusted (p-value corrected for multiple hypothesis testing) is less than 0.1.

We can supply additional arguments to the function to slightly adjust our plot:

<h2>Obtain shrunken log fold change values</h2>

As you can see, much like with our MA plot for our single replicate, genes that have a lower mean also exhibit a greater log fold change, giving the MA plot its characteristic arrowhead shape. This is due to the fact that noise in the measurements will lead to larger dispersion and a greater log fold change.

We can correct for this by calculating the shrunken log fold change. This allows us to better visualize genes whose differential expression is likely due to true biological or technical variance rather than noise. 

First, let's pull out the comparison groups from our <code>DESeqDataSet</code>:

Then, we can call up the <code>lfcShrink()</code> function to calculate our shrunken log fold change:

Now let's take a look at the results and export them for use later:

<h2>Plot MA plot for shrunken log fold change</h2>

Now let's take a look at how our MA plot looks like when we take into account that lower means are expected to have higher log fold change: